# Bluetti Elite 200 V2 — Register Exploration

Dataset: `data/snapshots.jsonl` — 1730 snapshots, ~45h (Apr 15–16 2026), 98.2% success.

**Plan:**
- **Step A** — Time series of all active registers (classify signal shapes)
- **Step B** — State-based comparison (group by reg 1509 output-state enum)
- **Step C** — Correlation against known anchors (SoC, AC power, AC input, DC input)

In [ ]:
import json
import struct
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 3)
plt.rcParams['figure.dpi'] = 110

SNAPSHOT_FILE = Path('../data/snapshots.jsonl')

def decode_block(hex_str):
    raw = bytes.fromhex(hex_str)
    return [struct.unpack('>H', raw[i:i+2])[0] for i in range(0, len(raw), 2)]

rows = []
with open(SNAPSHOT_FILE) as f:
    for line in f:
        d = json.loads(line)
        if not d['ok']:
            continue
        row = {'ts': pd.Timestamp(d['ts']), 'seq': d['seq']}
        for start_str, hex_str in d['registers'].items():
            start = int(start_str)
            for i, val in enumerate(decode_block(hex_str)):
                row[start + i] = val
        rows.append(row)

df = pd.DataFrame(rows).set_index('ts').sort_index()
print(f"Loaded {len(df)} snapshots from {df.index[0]} to {df.index[-1]}")
print(f"Registers: {len(df.columns) - 1} columns (including seq)")

## Active registers

Only registers with more than 1 unique value — everything else is a constant.

In [ ]:
reg_cols = [c for c in df.columns if isinstance(c, int)]
nunique = df[reg_cols].nunique()
active = nunique[nunique > 1].sort_index()

summary = pd.DataFrame({
    'unique_vals': active,
    'min': df[active.index].min(),
    'max': df[active.index].max(),
})
summary['range'] = summary['max'] - summary['min']
print(f"{len(active)} active registers:")
summary

## Step A — Time series of all active registers

One plot per register. Colour-coded by signal shape:
- Smooth / many unique values → physical measurement
- Few unique values (≤6) → enum / mode
- Monotonically increasing → counter

In [ ]:
# Known fields for reference labels
KNOWN = {
    102: 'BATTERY_SOC (%)',
    140: 'DC_OUTPUT_POWER (W)',
    142: 'AC_OUTPUT_POWER (W)',
    144: 'DC_INPUT_POWER (W)',
    146: 'AC_INPUT_POWER (W)',
    1314: 'AC_INPUT_VOLTAGE (×0.1V)',
}

def plot_reg(ax, reg, series, title_suffix=''):
    n_unique = series.nunique()
    color = 'steelblue' if n_unique > 6 else 'darkorange'
    ax.plot(series.index, series.values, color=color, linewidth=0.8)
    label = KNOWN.get(reg, f'reg {reg}')
    ax.set_title(f"{label}{title_suffix}  [unique={n_unique}, min={series.min()}, max={series.max()}]",
                 fontsize=9, loc='left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3)

regs_to_plot = sorted(active.index)
n = len(regs_to_plot)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n))
if n == 1:
    axes = [axes]

for ax, reg in zip(axes, regs_to_plot):
    plot_reg(ax, reg, df[reg].dropna())

plt.tight_layout(h_pad=1.5)
plt.savefig('../data/step_a_timeseries.png', bbox_inches='tight')
plt.show()
print("Saved to data/step_a_timeseries.png")

## Step A (focused) — Known + interesting unknowns side by side

Zoom into the registers most likely to be physically meaningful, on a shared time axis.

In [ ]:

# Registers we care most about — edit freely
FOCUS = [
    142,
    148,   # AC output current?
    1510,  # unknown
]
FOCUS = [r for r in FOCUS if r in df.columns]

fig, axes = plt.subplots(len(FOCUS), 1, figsize=(14, 3 * len(FOCUS)), sharex=True)
for ax, reg in zip(axes, FOCUS):
    plot_reg(ax, reg, df[reg].dropna())

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
plt.tight_layout(h_pad=1.5)
plt.savefig('../data/step_a_focused.png', bbox_inches='tight')
plt.show()
print("Saved to data/step_a_focused.png")


## Step B — State-based comparison

Reg 1509 is the output-state enum. Values seen: `[2, 3, 5]`.
Split dataset by state, compare register distributions.

In [ ]:
state_col = 1509
state_counts = df[state_col].value_counts().sort_index()
print("Reg 1509 value counts:")
print(state_counts)
print()

# Per-state stats for all active registers
state_stats = df.groupby(state_col)[active.index.tolist()].agg(['mean', 'min', 'max'])

# Show registers where mean differs notably across states
means = df.groupby(state_col)[active.index.tolist()].mean()
mean_spread = means.max() - means.min()
varying = mean_spread[mean_spread > 5].sort_values(ascending=False)
print("Registers with largest mean spread across output states:")
print(varying.to_string())

In [ ]:

# Plot state timeline + top varying registers together
top_varying = varying.head(12).index.tolist()
plot_regs = [state_col] + top_varying

fig, axes = plt.subplots(len(plot_regs), 1, figsize=(14, 3 * len(plot_regs)), sharex=True)
for ax, reg in zip(axes, plot_regs):
    plot_reg(ax, reg, df[reg].dropna())

plt.tight_layout(h_pad=1.5)
plt.savefig('../data/step_b_states.png', bbox_inches='tight')
plt.show()
print("Saved to data/step_b_states.png")


## Step C — Correlation against known anchors

In [ ]:
anchors = {
    'SoC (102)': 102,
    'AC_out (142)': 142,
    'AC_in (146)': 146,
    'DC_in/solar (144)': 144,
}

corr_results = {}
for label, anchor_reg in anchors.items():
    if anchor_reg not in df.columns:
        continue
    corr = df[active.index.tolist()].corrwith(df[anchor_reg]).abs().sort_values(ascending=False)
    corr_results[label] = corr

corr_df = pd.DataFrame(corr_results)
corr_df.index.name = 'register'

# Show top 20 for each anchor
for col in corr_df.columns:
    print(f"\nTop correlates with {col}:")
    top = corr_df[col].dropna().sort_values(ascending=False).head(20)
    for reg, val in top.items():
        label = KNOWN.get(reg, '')
        print(f"  reg {reg:5d}  r={val:.3f}  {label}")

In [ ]:

# Heatmap of correlations: anchors × top unknown registers
unknown_regs = [r for r in active.index if r not in KNOWN]
corr_unknowns = corr_df.loc[unknown_regs].dropna(how='all')
top_unknowns = corr_unknowns.max(axis=1).sort_values(ascending=False).head(30).index

fig, ax = plt.subplots(figsize=(8, 10))
data = corr_df.loc[top_unknowns].T
im = ax.imshow(data.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(top_unknowns)))
ax.set_xticklabels([f'r{r}' for r in top_unknowns], rotation=90, fontsize=8)
ax.set_yticks(range(len(data.index)))
ax.set_yticklabels(data.index, fontsize=9)
plt.colorbar(im, ax=ax, label='|correlation|')
ax.set_title('Correlation of unknown registers vs known anchors', fontsize=10)
plt.tight_layout()
plt.savefig('../data/step_c_correlation.png', bbox_inches='tight')
plt.show()
print("Saved to data/step_c_correlation.png")
